# 实验导航：HR 离职预测快速流程
- **步骤0** 导入依赖并配置画图主题。
- **步骤1** 读取 CSV、查看基本结构与缺失值。
- **步骤2** 明确目标列、数值列与类别列。
- **步骤3** 划分训练/测试集、标准化数值列并对类别列做独热编码。
- **步骤4** 训练逻辑回归基线模型，输出指标与混淆矩阵。
- **步骤5** 训练随机森林对照模型，展示成绩和特征重要度。
- **步骤6** 记录下一步延伸任务，方便课堂或作业拓展。

In [ ]:
# 步骤0：导入依赖并设置全局风格
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style(style="whitegrid")  # 统一图表风格，避免每次重复设置
DATA_PATH = Path("HR_comma_sep.csv")
assert DATA_PATH.exists(), f"未找到数据文件: {DATA_PATH}"

In [ ]:
# 步骤1：读取 HR 数据并查看样本/字段规模
df = pd.read_csv(DATA_PATH)
print(f"样本量: {df.shape[0]}, 字段数: {df.shape[1]}")
print(df.head())  # 预览前 5 行了解列名与取值范围

# 二元/离散字段不使用 KDE，避免旧版出现零带宽错误。
sns.pairplot(df, vars=["Work_accident", "average_montly_hours", "number_project"], hue="left", diag_kind="hist")
plt.show()

In [ ]:
# 步骤1补充：检查字段类型与缺失情况
df.info()
df.isna().sum().to_frame(name="missing")  # 缺失值全为 0，可直接建模

In [ ]:
# 步骤2：声明目标列与特征类型
# - 类别列按照题意固定：Work_accident、promotion_last_5years、sales、salary
# - 其余列全部作为数值特征，后续会进行标准化
target_col = "left"
categorical_cols = ["Work_accident", "promotion_last_5years", "sales", "salary"]
numeric_cols = [col for col in df.columns if col not in categorical_cols + [target_col]]
print("数值列:", numeric_cols)
print("类别列:", categorical_cols)

In [ ]:
# 步骤3：划分数据集并完成标准化 + 独热编码
X = df.drop(columns=[target_col]).copy()
y = df[target_col].astype(int)

# 3.1 分层抽样，保持离职比例一致
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3.2 仅对数值列做标准化，避免尺度差异影响线性模型
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# 3.3 对类别列做独热编码，不丢信息
dummy_kwargs = {"columns": categorical_cols, "drop_first": False}
X_train_ready = pd.get_dummies(X_train_scaled, **dummy_kwargs)
X_test_ready = pd.get_dummies(X_test_scaled, **dummy_kwargs)

# 3.4 对齐列：防止测试集缺少某些类别导致列不一致
X_train_ready, X_test_ready = X_train_ready.align(
    X_test_ready, join="left", axis=1, fill_value=0
)

print(X_train_ready.shape, X_test_ready.shape)

In [ ]:
# 步骤4：训练逻辑回归基线并输出指标
logit = LogisticRegression(max_iter=1000)
logit.fit(X_train_ready, y_train)
logit_pred = logit.predict(X_test_ready)

logit_acc = accuracy_score(y_test, logit_pred)
print(f"Logistic Regression Accuracy: {logit_acc:.3f}")
print(classification_report(y_test, logit_pred, digits=3))

# 直观展示误判位置，方便课堂讨论
cm = confusion_matrix(y_test, logit_pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
# 步骤5：训练随机森林并查看特征重要度
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"  # 调整类别权重，具体效果需验证
)
rf.fit(X_train_ready, y_train)
rf_pred = rf.predict(X_test_ready)
rf_acc = accuracy_score(y_test, rf_pred)
print(f"Random Forest Accuracy: {rf_acc:.3f}")
print(classification_report(y_test, rf_pred, digits=3))

feat_importance = (
    pd.Series(rf.feature_importances_, index=X_train_ready.columns)
    .sort_values(ascending=False)
    .head(10)
)
plt.figure(figsize=(6, 4))
sns.barplot(x=feat_importance.values, y=feat_importance.index)
plt.title("Top Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## 步骤6：延伸建议
- 将 `categorical_cols` 与 `numeric_cols` 替换为团队认可的字段列表，确保语义准确。
- 在逻辑回归中尝试 `class_weight`、SMOTE 或调参以提升少数类召回。
- 引入更多模型（XGBoost、LightGBM 等）并把指标、混淆矩阵、特征重要度截图写入实验报告。

### 自主调查
本 Notebook 仅演示两个模型的基本流程。请自行扩展第三种分类器，补充 ROC/AUC 与模型比较，并设计参数实验；结合自己的运行结果完成报告。参数选择应依据训练集内部的验证数据。
